# ChainCheck — DeBERTa-v3-small Fine-Tuning

Fine-tune **[microsoft/deberta-v3-small](https://huggingface.co/microsoft/deberta-v3-small)** on the
[HaluEval QA split](https://huggingface.co/datasets/pminervini/HaluEval) for **binary claim-level
hallucination detection** (label 0 = truthful, label 1 = hallucinated).

The fine-tuned model can drop in as a faster, cheaper alternative to the LLM judge inside
ChainCheck's `nli` method.

---

**Runtime:** GPU (T4 or better). In Google Colab: *Runtime → Change runtime type → T4 GPU*.  
**Expected wall-clock time:** ~40 min on T4 for 3 epochs over 10 000 training pairs.  
**Output:** `deberta-halueval/` — a HuggingFace `AutoModelForSequenceClassification` directory
you can push directly to the Hub or load with `from_pretrained`.


## 0 · Environment setup

In [ ]:
# Install / upgrade required packages
!pip install -q \
    transformers>=4.40 \
    datasets>=2.19 \
    accelerate>=0.29 \
    sentencepiece \
    protobuf \
    scikit-learn \
    evaluate \
    wandb          # optional — comment out if you don't want W&B logging

In [ ]:
import os, json, random, warnings
import numpy as np
import torch

from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1 · Configuration

Tweak `N_TRAIN` / `N_VAL` to trade speed for accuracy, or swap in a larger backbone.

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME     = "microsoft/deberta-v3-small"   # ~44 M params; swap for -base for +2 F1
OUTPUT_DIR     = "deberta-halueval"
HUB_MODEL_ID   = ""   # e.g. "youruser/chaincheck-deberta" — leave blank to skip push

# ── Data ───────────────────────────────────────────────────────────────────────
N_TRAIN        = 10_000   # training pairs  (max ~50 k from HaluEval QA)
N_VAL          = 2_000    # validation pairs
N_TEST         = 2_000    # held-out test pairs
MAX_LENGTH     = 384      # tokens; context+question+answer

# ── Training ───────────────────────────────────────────────────────────────────
EPOCHS         = 3
BATCH_SIZE     = 16       # reduce to 8 if OOM on T4
GRAD_ACCUM     = 2        # effective batch = 32
LR             = 2e-5
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.06
FP16           = device.type == "cuda"  # auto-enable mixed precision on GPU

# ── Labels ─────────────────────────────────────────────────────────────────────
LABEL2ID = {"truthful": 0, "hallucinated": 1}
ID2LABEL = {0: "truthful", 1: "hallucinated"}

print(json.dumps({
    "model": MODEL_NAME,
    "train": N_TRAIN, "val": N_VAL, "test": N_TEST,
    "epochs": EPOCHS, "batch": BATCH_SIZE * GRAD_ACCUM,
    "fp16": FP16,
}, indent=2))

## 2 · Load and prepare HaluEval

In [ ]:
raw_ds = load_dataset("pminervini/HaluEval", "qa", split="data")
print(f"HaluEval QA rows: {len(raw_ds):,}")
print("Columns:", raw_ds.column_names)
raw_ds[0]

In [ ]:
def build_samples(ds, n_max: int) -> list[dict]:
    """
    Convert HaluEval rows into flat (text, label) samples.

    Each row contributes:
      • one truthful  sample  (right_answer,        label=0)
      • one hallucinated sample (hallucinated_answer, label=1)

    Input text format:  "question: {q}\ncontext: {c}\nanswer: {a}"
    This matches the format ChainCheck's NLI method will use at inference time.
    """
    samples = []
    for row in ds:
        if len(samples) >= n_max:
            break
        q = (row.get("question") or "").strip()
        c = (row.get("knowledge") or "").strip()
        right = (row.get("right_answer") or "").strip()
        wrong = (row.get("hallucinated_answer") or "").strip()

        if right and len(samples) < n_max:
            samples.append({
                "text": f"question: {q}\ncontext: {c}\nanswer: {right}",
                "label": 0,
            })
        if wrong and len(samples) < n_max:
            samples.append({
                "text": f"question: {q}\ncontext: {c}\nanswer: {wrong}",
                "label": 1,
            })
    random.shuffle(samples)
    return samples


total_needed = N_TRAIN + N_VAL + N_TEST
all_samples  = build_samples(raw_ds, total_needed)
print(f"Total samples built: {len(all_samples):,}")
print(f"Class balance: {sum(s['label']==1 for s in all_samples)/len(all_samples):.1%} hallucinated")

In [ ]:
train_samples = all_samples[:N_TRAIN]
val_samples   = all_samples[N_TRAIN : N_TRAIN + N_VAL]
test_samples  = all_samples[N_TRAIN + N_VAL : N_TRAIN + N_VAL + N_TEST]

train_ds = Dataset.from_list(train_samples)
val_ds   = Dataset.from_list(val_samples)
test_ds  = Dataset.from_list(test_samples)

ds = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})
print(ds)

## 3 · Tokenise

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenised = ds.map(tokenize, batched=True, remove_columns=["text"])
tokenised.set_format("torch")
print(tokenised)
print("Sample token count:", tokenised["train"][0]["input_ids"].shape)

## 4 · Load model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params/1e6:.1f} M total, {n_train/1e6:.1f} M trainable")

## 5 · Train

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, average="binary"),
        "precision": precision_score(labels, preds, average="binary", zero_division=0),
        "recall":    recall_score(labels, preds, average="binary"),
        "roc_auc":   roc_auc_score(labels, probs),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    fp16                        = FP16,
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = "wandb" if os.getenv("WANDB_API_KEY") else "none",
    run_name                    = "chaincheck-deberta-halueval",
    seed                        = SEED,
    dataloader_num_workers      = 2,
    save_total_limit            = 2,
)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenised["train"],
    eval_dataset    = tokenised["validation"],
    tokenizer       = tokenizer,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 6 · Evaluate on held-out test set

In [ ]:
results = trainer.evaluate(tokenised["test"])
print("\n── Test-set results ──")
for k, v in results.items():
    if k.startswith("eval_"):
        print(f"  {k[5:]:12s}  {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

preds_out   = trainer.predict(tokenised["test"])
y_pred      = np.argmax(preds_out.predictions, axis=-1)
y_true      = preds_out.label_ids

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["truthful", "hallucinated"])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("DeBERTa-v3-small — HaluEval QA (test)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print(f"Test F1: {f1_score(y_true, y_pred):.4f}")

## 7 · Save model

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}/")

In [ ]:
# Optional: push to HuggingFace Hub
# Fill in HUB_MODEL_ID at the top of this notebook, then run this cell.

if HUB_MODEL_ID:
    from huggingface_hub import notebook_login
    notebook_login()   # prompts for token
    trainer.push_to_hub(HUB_MODEL_ID)
    tokenizer.push_to_hub(HUB_MODEL_ID)
    print(f"Pushed to https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("HUB_MODEL_ID not set — skipping Hub push.")

## 8 · Inference demo

After training, load the saved model and run a quick sanity check.
This is exactly how ChainCheck would call the model as a drop-in NLI replacement.

In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if device.type == "cuda" else -1,
    truncation=True,
    max_length=MAX_LENGTH,
)

EXAMPLES = [
    {
        "label": "truthful",
        "text": (
            "question: What is the capital of France?\n"
            "context: France is a country in Western Europe.\n"
            "answer: The capital of France is Paris."
        ),
    },
    {
        "label": "hallucinated",
        "text": (
            "question: What is the capital of France?\n"
            "context: France is a country in Western Europe.\n"
            "answer: The capital of France is Lyon."
        ),
    },
    {
        "label": "truthful",
        "text": (
            "question: Who wrote Hamlet?\n"
            "context: Hamlet is an Elizabethan play published around 1600.\n"
            "answer: Hamlet was written by William Shakespeare."
        ),
    },
    {
        "label": "hallucinated",
        "text": (
            "question: Who wrote Hamlet?\n"
            "context: Hamlet is an Elizabethan play published around 1600.\n"
            "answer: Hamlet was written by Christopher Marlowe."
        ),
    },
]

print(f"{'Expected':<14} {'Predicted':<14} {'Score':>6}  Text snippet")
print("-" * 70)
for ex in EXAMPLES:
    out = clf(ex["text"])[0]
    snippet = ex["text"].split("\n")[-1][:40]
    match = "✓" if out["label"].lower() == ex["label"] else "✗"
    print(f"{ex['label']:<14} {out['label']:<14} {out['score']:>6.3f}  {snippet}  {match}")

## 9 · Integrating with ChainCheck

Once the model is saved (or pushed to the Hub), swap it into ChainCheck's NLI method
by setting the environment variable before starting the server:

```bash
# Local saved model
export CHAINCHECK_NLI_MODEL=./deberta-halueval

# Or a Hub model
export CHAINCHECK_NLI_MODEL=youruser/chaincheck-deberta

uvicorn chaincheck.server:app --reload
```

The `nli` method in `chaincheck/methods/nli.py` reads `CHAINCHECK_NLI_MODEL` and
loads the model via `AutoModelForSequenceClassification.from_pretrained()`.

### Expected benchmark uplift

| Model                       | HaluEval F1 | Latency (ms/claim) |
|-----------------------------|:-----------:|:------------------:|
| cross-encoder/nli-deberta-v3-small (baseline) | 0.788 | ~80 |
| **DeBERTa-v3-small fine-tuned** (this notebook) | **~0.83–0.86** | ~80 |
| DeBERTa-v3-base fine-tuned   | ~0.87–0.89 | ~140 |

*Estimated — actual numbers depend on your hyperparameters and random seed.*

---

### Colab tips

- **OOM on T4?** Reduce `BATCH_SIZE` to 8 (effective batch stays at 32 via grad accumulation).
- **Want faster iteration?** Set `N_TRAIN=2000`, `EPOCHS=2` to do a quick smoke-test in <10 min.
- **Save to Drive**: add a cell with `from google.colab import drive; drive.mount('/content/drive')`
  and change `OUTPUT_DIR` to a path under `/content/drive/MyDrive/`.
